# Prep

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from tqdm import tqdm
from run_refinement import load_model, load_inputs, prepare_inputs, eval_depth, visualize_output, compute_ause, compute_aurg, MSPN
from argparse import Namespace

In [ ]:
data_dir = os.path.join('..','data')

#train_dir = os.path.join(data_dir, 'train', 'train')
test_dir = os.path.join(data_dir, 'test', 'test')

rgb_dir = os.path.join(data_dir, 'train','train')
rgb_test_dir = os.path.join(data_dir, 'test','test')
depth_md_dir = os.path.join(data_dir,'mean_train')
total_var_train_dir = os.path.join(data_dir,'total_var_train')
guidance_dir = os.path.join(data_dir, 'guidance_train')

train_list_file = os.path.join(data_dir, 'train_list.txt')
test_list_file = os.path.join(data_dir, 'test_list.txt')

output_dir = os.path.join(data_dir, 'output')
results_dir = os.path.join(output_dir, 'results')
predictions_dir = os.path.join(output_dir, 'predictions')

### Hyperparameters

In [ ]:
EPS = 1e-8

BATCH_SIZE = 4
LEARNING_RATE = 1e-2
WEIGHT_DECAY = 1e-2
NUM_EPOCHS = 36
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
INPUT_SIZE = (426, 560)
NUM_WORKERS = 4
PIN_MEMORY = True
VAR_THRESHOLD = 0.05
BETA = 0.5
EMBED_DIM = 64
PROP_TIME = 6


args = Namespace(
    data_name='NYU',
    mode='SDR',
    embed_dim=EMBED_DIM,
    pretrain=os.path.join('.', 'MSPN_SDR', 'test_models', 'SDR_NYU.pt'),
    prop_time=PROP_TIME,
)

In [ ]:
guidance_net = torch.load(os.path.join('.', 'MSPN_SDR', 'test_models', 'guidance_net.pt'))
guidance_net = guidance_net.to(DEVICE)
guidance_net = guidance_net.eval()
#guidance_net = torch.compile(guidance_net, mode='max-autotune')

### Helper functions

In [ ]:
def ensure_dir(directory):
    if not os.path.exists(directory):
        os.makedirs(directory)

def target_transform(depth):
    # Resize the depth map to match input size
    depth = torch.nn.functional.interpolate(
        depth.unsqueeze(0).unsqueeze(0), 
        size=INPUT_SIZE, 
        mode='bilinear', 
        align_corners=True
    ).squeeze()
    
    # Add channel dimension to match model output
    depth = depth.unsqueeze(0)
    return depth

def beta_nll_loss(mean, target, variance, beta=0.5, eps=EPS, reduction='mean'):
    """Compute beta-NLL loss
    
    :param mean: Predicted mean of shape B x D
    :param variance: Predicted variance of shape B x D
    :param target: Target of shape B x D
    :param beta: Parameter from range [0, 1] controlling relative 
        weighting between data points, where `0` corresponds to 
        high weight on low error points and `1` to an equal weighting.
    :returns: Loss per batch element of shape B
    """
    eps_variance = variance + eps
    loss = 0.5 * ((target - mean) ** 2 / eps_variance + eps_variance.log())

    if beta > 0:
        loss = loss * (eps_variance.detach() ** beta)
    
    if reduction == 'mean':
        return loss.mean()  # Average over all elements (matches GaussianNLLLoss default)
    elif reduction == 'sum':
        return loss.sum()   # Sum over all elements
    elif reduction == 'none':
        return loss        # Keep all elements (shape [B, 1, H, W])
    else:
        raise ValueError(f"Invalid reduction: {reduction}")

def silog_loss(pred,
               target,
               eps = EPS,
               reduction = 'mean'):
    """Computes the Scale-Invariant Logarithmic (SI-Log) loss between
    prediction and target.

    Args:
        pred (Tensor): Predicted output.
        target (Tensor): Ground truth.
        weight (Optional[Tensor]): Optional weight to apply on the loss.
        eps (float): Epsilon value to avoid division and log(0).
        reduction (Union[str, None]): Specifies the reduction to apply to the
            output: 'mean', 'sum' or None.
        avg_factor (Optional[int]): Optional average factor for the loss.

    Returns:
        Tensor: The calculated SI-Log loss.
    """
    pred, target = pred.flatten(1), target.flatten(1)
    valid_mask = (target > eps).detach().float()

    diff_log = torch.log(target.clamp(min=eps)) - torch.log(
        pred.clamp(min=eps))

    valid_mask = (target > eps).detach() & (~torch.isnan(diff_log))
    diff_log[~valid_mask] = 0.0
    valid_mask = valid_mask.float()

    diff_log_sq_mean = (diff_log.pow(2) * valid_mask).sum(
        dim=1) / valid_mask.sum(dim=1).clamp(min=eps)
    diff_log_mean = (diff_log * valid_mask).sum(dim=1) / valid_mask.sum(
        dim=1).clamp(min=eps)

    loss = torch.sqrt(diff_log_sq_mean - 0.5 * diff_log_mean.pow(2))

    if reduction == 'mean':
        return loss.mean()  # Average over all elements (matches GaussianNLLLoss default)
    elif reduction == 'sum':
        return loss.sum()   # Sum over all elements
    elif reduction == 'none':
        return loss        # Keep all elements (shape [B, 1, H, W])
    else:
        raise ValueError(f"Invalid reduction: {reduction}")

def CustomLoss(mean, target, variance, silog_weight=0.7):
    return silog_loss(mean, target)
    #return silog_weight * silog_loss(mean, target) + (1-silog_weight) * beta_nll_loss(mean, target, variance)

# Dataset

In [ ]:
def get_file_tuple(sample_num):
    num = sample_num.strip()
    return [
            f'sample_{num}_rgb.png',
            f'sample_{num}_depth.npy',
            f'sample_{num}_depth_mean.npy',
            f'sample_{num}_depth_var_total.npy',
            ]

class DepthDataset(Dataset):
    def __init__(self, data_dir, list_file, transform=None, target_transform=None, has_gt=True):
        self.data_dir = data_dir
        self.transform = transform
        self.target_transform = target_transform
        self.has_gt = has_gt
        
        # Read file list
        with open(list_file, 'r') as f:
            if has_gt:
                #chosen_idx = np.random.choice(23971, 640, replace=False)
                #file = np.array(list(f))
                #self.file_tuples = [get_file_tuple(line) for line in file[chosen_idx]]
                #print(file[chosen_idx])
                self.file_tuples = [get_file_tuple(line) for line in f]
            else:
                # For test set without ground truth
                self.file_list = [line.strip() for line in f]
    
    def __len__(self):
        return len(self.file_tuples if self.has_gt else self.file_list)
    
    def __getitem__(self, idx):
        if self.has_gt:
            rgb_path = os.path.join(rgb_dir, self.file_tuples[idx][0])
            depth_path = os.path.join(rgb_dir, self.file_tuples[idx][1])
            depth_md_path = os.path.join(depth_md_dir, self.file_tuples[idx][2])
            total_var_path = os.path.join(total_var_train_dir, self.file_tuples[idx][3])

            rgb, s_depth, depth_md, variance = load_inputs(rgb_path, total_var_path, depth_md_path, threshold=VAR_THRESHOLD)
            
            # Load RGB image
            rgb = Image.open(rgb_path).convert('RGB')
            
            # Load depth map
            depth = np.load(depth_path).astype(np.float32)
            depth = torch.from_numpy(depth)
            
            # Apply transformations
            if self.transform:
                rgb = self.transform(rgb)
            
            if self.target_transform:
                depth = self.target_transform(depth)
            else:
                # Add channel dimension if not done by transform
                depth = depth.unsqueeze(0)

            return rgb, depth, s_depth, depth_md, variance, self.file_tuples[idx][0]  # Return filename for saving predictions
        else:
            # For test set without ground truth
            rgb_path = os.path.join(rgb_test_dir, self.file_list[idx].split(' ')[0])
            
            # Load RGB image
            rgb = Image.open(rgb_path).convert('RGB')
            
            # Apply transformations
            if self.transform:
                rgb = self.transform(rgb)
            
            return rgb, self.file_list[idx]  # No depth, just return the filename

# Training loop

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs, device):
    """Train the model
    and save the best based on validation metrics"""
    best_val_loss = float('inf')
    best_epoch = 0
    train_losses = []
    val_losses = []

    torch.backends.cudnn.benchmark = True
    scaler = torch.cuda.amp.GradScaler()

    for epoch in range(num_epochs):
        print(f"Epoch {epoch+1}/{num_epochs}")
        
        # Training phase
        model.train()
        train_loss = 0.0
        
        for inputs, targets, s_depth, depth_md, variance, filenames in tqdm(train_loader, desc="Training"):
            inputs = inputs.to(device)
            targets = targets.to(device)
            pred_init, guide, s_depth, var_init = prepare_inputs(inputs, s_depth, depth_md, variance, guidance_net, device)
            optimizer.zero_grad()

            y_inter, _, _ = model(
                pred_init,
                var_init,
                guide,
            )
            loss = torch.tensor(0, device=device, dtype=float)
            for yy in y_inter[1:]:
                loss += criterion(yy, targets)

            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * inputs.size(0)

        train_loss /= len(train_loader.dataset)
        train_losses.append(train_loss)

        # Validation phase
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for inputs, targets, s_depth, depth_md, variance, filenames in tqdm(val_loader, desc="Validation"):
                inputs = inputs.to(device)
                targets = targets.to(device)

                pred_init, guide, s_depth, var_init = prepare_inputs(inputs, s_depth, depth_md, variance, guidance_net, device)
                
                # Forward pass
                y_inter, _, _ = model(
                    pred_init,
                    var_init,
                    guide,
                )

                loss = torch.tensor(0, device=device, dtype=float)
                for yy in y_inter[1:]:
                    loss += criterion(yy, targets)
                
                val_loss += loss.item() * inputs.size(0)
        
        val_loss /= len(val_loader.dataset)
        #val_loss /= 3
        val_losses.append(val_loss)
                
        print(f"Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}")
        
        # Save the best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            torch.save(model.state_dict(), os.path.join(results_dir, 'best_model.pth'))
            print(f"New best model saved at epoch {epoch+1} with validation loss: {val_loss:.4f}")
    
    print(f"\nBest model was from epoch {best_epoch+1} with validation loss: {best_val_loss:.4f}")
    
    # Load the best model
    model.load_state_dict(torch.load(os.path.join(results_dir, 'best_model.pth')))
    
    return model

# Model evaluation

In [ ]:
def evaluate_model(model, val_loader, device):
    """Evaluate the model and compute metrics on validation set"""
    model.eval()
    
    val_loss = 0.0
    
    mae = 0.0
    rmse = 0.0
    rel = 0.0
    delta1 = 0.0
    delta2 = 0.0
    delta3 = 0.0
    sirmse = 0.0

    mae = np.zeros(PROP_TIME+1)
    rmse = np.zeros(PROP_TIME+1)
    rel = np.zeros(PROP_TIME+1)
    delta1 = np.zeros(PROP_TIME+1)
    delta2 = np.zeros(PROP_TIME+1)
    delta3 = np.zeros(PROP_TIME+1)
    sirmse = np.zeros(PROP_TIME+1)
    
    total_samples = 0
    target_shape = None
    
    with torch.no_grad():
        for inputs, targets, s_depth, depth_md, variance, filenames in tqdm(val_loader, desc="Evaluating"):
        #for inputs, targets, filenames in tqdm(val_loader, desc="Evaluating"):
            inputs = inputs.to(device)
            targets = targets.to(device)
            pred_init, guide, s_depth, var_init = prepare_inputs(inputs, s_depth, depth_md, variance, guidance_net, device)
            
            batch_size = inputs.size(0)
            total_samples += batch_size
            
            if target_shape is None:
                target_shape = targets.shape

            #outputs = model(inputs)
            y_inter, _, _ = model(
                pred_init,
                var_init,
                guide,
            )

            # Resize outputs to match target dimensions
            loss = torch.tensor(0, device=device, dtype=float)
            for yy in y_inter[1:]:
                loss += silog_loss(yy, targets)
            val_loss += loss.item() * inputs.size(0)
            
            
            for j, y in enumerate(y_inter):
                outputs = nn.functional.interpolate(
                    y,
                    size=targets.shape[-2:],  # Match height and width of targets
                    mode='bilinear',
                    align_corners=True
                )

                abs_diff = torch.abs(outputs - targets)
                mae[j] += torch.sum(abs_diff).item()
                rmse[j] += torch.sum(torch.pow(abs_diff, 2)).item()
                rel[j] += torch.sum(abs_diff / (targets + 1e-6)).item()
                
                # Calculate scale-invariant RMSE for each image in the batch
                for i in range(batch_size):
                    # Convert tensors to numpy arrays
                    pred_np = outputs[i].cpu().squeeze().numpy()
                    target_np = targets[i].cpu().squeeze().numpy()
                    
                    EPSILON = 1e-6
                    
                    valid_target = target_np > EPSILON
                    if not np.any(valid_target):
                        continue
                    
                    target_valid = target_np[valid_target]
                    pred_valid = pred_np[valid_target]
                    
                    log_target = np.log(target_valid)
                    
                    pred_valid = np.where(pred_valid > EPSILON, pred_valid, EPSILON)
                    log_pred = np.log(pred_valid)
                    
                    # Calculate scale-invariant error
                    diff = log_pred - log_target
                    diff_mean = np.mean(diff)
                    
                    # Calculate RMSE for this image
                    sirmse[j] += np.sqrt(np.mean((diff - diff_mean) ** 2))
                
                # Calculate thresholded accuracy
                max_ratio = torch.max(outputs / (targets + 1e-6), targets / (outputs + 1e-6))
                delta1[j] += torch.sum(max_ratio < 1.25).item()
                delta2[j] += torch.sum(max_ratio < 1.25**2).item()
                delta3[j] += torch.sum(max_ratio < 1.25**3).item()
                
                # Save some sample predictions
                if total_samples <= 5 * batch_size:
                    for i in range(min(batch_size, 5)):
                        idx = total_samples - batch_size + i
                        
                        # Convert tensors to numpy arrays
                        input_np = inputs[i].cpu().permute(1, 2, 0).numpy()
                        target_np = targets[i].cpu().squeeze().numpy()
                        output_np = outputs[i].cpu().squeeze().numpy()
                        
                        # Normalize for visualization
                        input_np = (input_np - input_np.min()) / (input_np.max() - input_np.min() + 1e-6)
                        
                        # Create visualization
                        plt.figure(figsize=(15, 5))
                        
                        plt.subplot(1, 3, 1)
                        plt.imshow(input_np)
                        plt.title("RGB Input")
                        plt.axis('off')
                        
                        plt.subplot(1, 3, 2)
                        plt.imshow(target_np, cmap='plasma')
                        plt.title("Ground Truth Depth")
                        plt.axis('off')
                        
                        plt.subplot(1, 3, 3)
                        plt.imshow(output_np, cmap='plasma')
                        plt.title("Predicted Depth")
                        plt.axis('off')
                        
                        plt.tight_layout()
                        plt.savefig(os.path.join(results_dir, f"sample_{idx}.png"))
                        plt.close()
                
                # Free up memory
                #del inputs, targets, outputs, abs_diff, max_ratio
            
        # Clear CUDA cache
        torch.cuda.empty_cache()
    
    # Calculate final metrics using stored target shape
    total_pixels = target_shape[1] * target_shape[2] * target_shape[3]  # channels * height * width
    mae /= total_samples * total_pixels
    rmse = np.sqrt(rmse / (total_samples * total_pixels))
    rel /= total_samples * total_pixels
    sirmse = sirmse / total_samples
    delta1 /= total_samples * total_pixels
    delta2 /= total_samples * total_pixels
    delta3 /= total_samples * total_pixels
    val_loss /= len(val_loader.dataset)
    print("Validation Loss: ", val_loss)
    metrics = {
        'MAE': mae,
        'RMSE': rmse,
        'siRMSE': sirmse,
        'REL': rel,
        'Delta1': delta1,
        'Delta2': delta2,
        'Delta3': delta3
    }
    
    return metrics

# Generate test predictions

In [ ]:
def generate_test_predictions(model, test_loader, device):
    """Generate predictions for the test set without ground truth"""
    model.eval()
    
    # Ensure predictions directory exists
    ensure_dir(predictions_dir)
    
    with torch.no_grad():
        
        for inputs, filenames in tqdm(test_loader, desc="Generating Test Predictions"):
            inputs = inputs.to(device)
            batch_size = inputs.size(0)
            
            # Forward pass
            outputs = model(inputs)
            
            # Resize outputs to match original input dimensions (426x560)
            outputs = nn.functional.interpolate(
                outputs,
                size=(426, 560),  # Original input dimensions
                mode='bilinear',
                align_corners=True
            )
            
            # Save all test predictions
            for i in range(batch_size):
                # Get filename without extension
                filename = filenames[i].split(' ')[1]
                
                # Save depth map prediction as numpy array
                depth_pred = outputs[i].cpu().squeeze().numpy()
                np.save(os.path.join(predictions_dir, f"{filename}"), depth_pred)
            
            # Clean up memory
            del inputs, outputs
        
        # Clear cache after test predictions
        torch.cuda.empty_cache()

# Putting it all together

In [ ]:
# Set a fixed random seed for reproducibility
np.random.seed(0)
torch.manual_seed(0)
torch.cuda.manual_seed_all(0)
#torch.use_deterministic_algorithms(True)
torch.backends.cudnn.deterministic = True

# Create output directories
ensure_dir(results_dir)
ensure_dir(predictions_dir)

# Define transforms
train_transform = transforms.Compose([
    transforms.Resize(INPUT_SIZE),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),  # Data augmentation
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize(INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create training dataset with ground truth
train_full_dataset = DepthDataset(
    data_dir=rgb_dir,
    list_file=train_list_file, 
    transform=train_transform,
    target_transform=target_transform,
    has_gt=True
)

# Create test dataset without ground truth
test_dataset = DepthDataset(
    data_dir=test_dir,
    list_file=test_list_file,
    transform=test_transform,
    has_gt=False  # Test set has no ground truth
)

print(f"Memory allocated after data init: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

# Split training dataset into train and validation
total_size = len(train_full_dataset)
train_size = int(0.80 * total_size)  # 80% for training
val_size = total_size - train_size    # 20% for validation



train_dataset, val_dataset = torch.utils.data.random_split(
    train_full_dataset, [train_size, val_size]
)

# Create data loaders with memory optimizations
train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=NUM_WORKERS, 
    pin_memory=PIN_MEMORY,
    drop_last=True,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    num_workers=NUM_WORKERS, 
    pin_memory=PIN_MEMORY
)

In [ ]:
print(f"Train size: {len(train_dataset)}, Validation size: {len(val_dataset)}, Test size: {len(test_dataset)}")

# Clear CUDA cache before model initialization
torch.cuda.empty_cache()

# Display GPU memory info
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Total GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"Initially allocated: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

model = MSPN(args)
#load_model(model, args)
#model = nn.DataParallel(model)
model = model.to(DEVICE)

print(f"Using device: {DEVICE}")

# Print memory usage after model initialization
if torch.cuda.is_available():
    print(f"Memory allocated after model init: {torch.cuda.memory_allocated(0) / 1e9:.2f} GB")

# Define loss function and optimizer
criterion = silog_loss
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, betas=[0.9, 0.999])

# Train the model
print("Starting training...")
model = train_model(model, train_loader, val_loader, criterion, optimizer, NUM_EPOCHS, DEVICE)

In [ ]:
import random
random.seed(0)
# Evaluate the model on validation set
print("Evaluating model on validation set...")

model = MSPN(args)
model = model.to(DEVICE)
model.load_state_dict(torch.load(os.path.join(results_dir, 'best_model.pth')))

metrics = evaluate_model(model, val_loader, DEVICE)

# Print metrics
print("\nValidation Metrics:")
for i in range(PROP_TIME+1):
    print(f"Iteration {i}")
    for name, value in metrics.items():
        print(f"{name}: {value[i]:.4f}")
    print()

    # Save metrics to file
    with open(os.path.join(results_dir, f'validation_metrics_iter.txt'), 'w') as f:
        for i in range(PROP_TIME+1):
            f.write(f"Iteration {i}\n")
            for name, value in metrics.items():
                f.write(f"{name}: {value[i]:.4f}\n")
            f.write("\n")

metrics = evaluate_model(model, train_loader, DEVICE)

# Print metrics
print("\nTraining Metrics:")
for i in range(PROP_TIME+1):
    print(f"Iteration {i}")
    for name, value in metrics.items():
        print(f"{name}: {value[i]:.4f}")
    print()

    # Save metrics to file
    with open(os.path.join(results_dir, f'train_metrics_iter.txt'), 'w') as f:
        for i in range(PROP_TIME+1):
            f.write(f"Iteration {i}\n")
            for name, value in metrics.items():
                f.write(f"{name}: {value[i]:.4f}\n")
            f.write("\n")

# Generate predictions for the test set
#print("Generating predictions for test set...")
#generate_test_predictions(model, test_loader, DEVICE)

print(f"Results saved to {results_dir}")
print(f"All test depth map predictions saved to {predictions_dir}")

In [ ]:
model = MSPN(args)
model = model.to(DEVICE)
model.load_state_dict(torch.load(os.path.join(results_dir, 'best_model.pth')))
model.eval()
# AUSE
# AURG
# Kalman Gain
# Variance maps
# Depth Predictions
with torch.no_grad():
    for inputs, targets, s_depth, depth_md, variance, filenames in tqdm(val_loader, desc="Evaluating"):
        inputs = inputs.to(DEVICE)
        targets = targets.to(DEVICE)
        pred_init, guide, s_depth, var_init = prepare_inputs(inputs, s_depth, depth_md, variance, guidance_net, DEVICE)    

        #outputs = model(inputs)
        y_inter, var_inter, gain_inter = model(
            pred_init,
            var_init,
            guide,
        )
        data = []
        for i in range(len(y_inter)):
            diff = y_inter[i] - targets
            mse = diff.square()
            iter_data = eval_depth(y_inter[i], targets)
            iter_data['ause'] = compute_ause(mse.squeeze(0).flatten().cpu().numpy(), var_inter[i].squeeze(0).flatten().cpu().numpy())
            iter_data['aurg'] = compute_aurg(mse.squeeze(0).flatten().cpu().numpy(), var_inter[i].squeeze(0).flatten().cpu().numpy())
            iter_data['var'] = var_inter[i].cpu().numpy()
            iter_data['pred'] = y_inter[i].cpu().numpy()
            if i == 0:
                iter_data['gain'] = torch.zeros_like(gain_inter[0]).cpu().numpy()
            else:
                iter_data['gain'] = gain_inter[i-1].cpu().numpy()
            #output = {'pred_inter': y_inter, 'var_inter': var_inter,
            #        'guidance': guide}
            #visualize_output(output, filenames[0][7:13])
            data.append(iter_data)
        
        np.save(os.path.join(results_dir, f'{filenames[0][7:13]}_eval_data.npy'), np.array(data))




In [ ]:
all_data = {}
for file in os.scandir(os.path.join(results_dir, 'epoch36_6iter_512imgs')):
    if file.name[-3:] == 'npy':
        all_data[file.name[:6]] = np.load(file.path, allow_pickle=True)

In [ ]:
idx = 0
for sample, data in all_data.items():
    if idx >= 5:
        break
    output = {'pred_inter': [iter['pred'] for iter in data], 'var_inter': [iter['var'] for iter in data]}
    print(sample)
    visualize_output(output, sample)
    idx += 1



In [ ]:
# Open a sample prediction from validation set
#Image.open(os.path.join(results_dir, 'sample_0.png'))